# M8: OpenSearch Semantic Search — Video Clip Retrieval

**Pipeline Position:** Stage 4: Search and Indexing (OpenSearch — AWS-native Cosmos Dataset Search)  
**Input S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m2/captions.json`  
**Output S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m8/`  
**Instance:** ml.t3.medium (CPU — no GPU required)  

> **Note:** This replaces NVIDIA Cosmos-embed NIM which requires an NVAIE (NVIDIA AI Enterprise) license.  
> OpenSearch Serverless with k-NN provides equivalent semantic search capability using AWS-native services.

## What This Module Does

1. Reads captions generated by M2 (Cosmos Reason Captioning)
2. Generates vector embeddings from caption text
3. Creates an OpenSearch Serverless k-NN vector index
4. Demonstrates natural language query → ranked video clip retrieval

## Cost Note

OpenSearch Serverless has a minimum cost of **~$24/month** when a collection is active (0.5 OCU indexing + 0.5 OCU search minimum). For workshop purposes, we use a time-limited collection and delete after demo.

In [ ]:
"""Environment Setup"""
import os
import json
import time
import re
from datetime import datetime, timezone

import boto3
import numpy as np

# --- S3 Path Configuration ---
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
# USER_PROFILE is injected by the provisioning LCC. If it's missing we must NOT
# silently fall back to "default": that would create an AOSS collection
# (av30-semantic-default) whose name never matches this user's teardown
# (av30-semantic-{user_id[:8]}), guaranteeing an orphaned, billing collection.
PROFILE = os.environ.get("USER_PROFILE", "").strip()
if not PROFILE:
    raise RuntimeError(
        "USER_PROFILE is not set in this kernel. It is normally injected by the "
        "workspace provisioning script. Without it, M8 cannot name its OpenSearch "
        "collection safely (a 'default' collection would be orphaned and keep "
        "billing). Restart the JupyterLab app so the launch config re-injects the "
        "environment, or set USER_PROFILE to your workshop user id, then re-run."
    )

USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")
INPUT_PREFIX = f"users/{PROFILE}/m2/"
OUTPUT_PREFIX = f"users/{PROFILE}/m8/"

# OpenSearch configuration. AOSS collection names must be 3-32 chars, lowercase
# letters/digits/hyphens, and start with a letter — sanitize the profile so an
# id with '_'/'.'/uppercase can't produce an invalid (or silently mangled) name.
def _aoss_name(profile):
    slug = re.sub(r"[^a-z0-9-]", "-", profile.lower())[:8].strip("-") or "user"
    if not slug[0].isalpha():
        slug = f"u{slug}"[:8]
    name = f"av30-semantic-{slug}"
    return name

COLLECTION_NAME = _aoss_name(PROFILE)
INDEX_NAME = "av-video-clips"
EMBEDDING_DIM = 384  # sentence-transformers/all-MiniLM-L6-v2 dimension

# --- Clients ---
s3 = boto3.client("s3")
aoss_client = boto3.client("opensearchserverless")

print(f"Account ID: {ACCOUNT_ID}")
print(f"Profile: {PROFILE}")
print(f"Input: s3://{USER_BUCKET}/{INPUT_PREFIX}")
print(f"Output: s3://{USER_BUCKET}/{OUTPUT_PREFIX}")
print(f"Collection: {COLLECTION_NAME}")
print(f"\nSetup complete.")

In [ ]:
"""Install dependencies — sentence-transformers for embedding, opensearch-py for client"""
import os
# transformers must NOT load the TensorFlow backend (SMD GPU image has
# Keras 3, which the transformers TF path rejects). Force the torch-only
# path before any transformers import.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
import subprocess
import sys

# pip name -> importable module name (they differ for opensearch-py and
# requests-aws4auth; probing the pip name would reinstall every run).
packages = {
    "sentence-transformers": "sentence_transformers",
    "opensearch-py": "opensearchpy",
    "requests-aws4auth": "requests_aws4auth",
}

import importlib.util
for pkg, module in packages.items():
    if importlib.util.find_spec(module) is not None:
        print(f"  {pkg}: already installed")
    else:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

from sentence_transformers import SentenceTransformer
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

print("\nAll dependencies ready.")

In [ ]:
"""Load M2 captions — these are the documents we will index"""
from botocore.exceptions import ClientError

captions_key = f"{INPUT_PREFIX}captions.json"
print(f"Loading captions from: s3://{USER_BUCKET}/{captions_key}")

try:
    response = s3.get_object(Bucket=USER_BUCKET, Key=captions_key)
except ClientError as e:
    code = e.response.get("Error", {}).get("Code", "")
    if code in ("NoSuchKey", "404", "NoSuchBucket", "AccessDenied"):
        raise RuntimeError(
            f"M2 captions not found at s3://{USER_BUCKET}/{captions_key} "
            f"({code}). Run M2 (Cosmos Reason Captioning) first — it writes "
            f"captions.json that M8 indexes."
        ) from e
    raise

m2_output = json.loads(response["Body"].read())
captions = m2_output.get("captions")
if not captions:
    raise RuntimeError(
        f"M2 output at s3://{USER_BUCKET}/{captions_key} has no 'captions' — "
        f"re-run M2 to regenerate it."
    )
print(f"Loaded {len(captions)} captions from M2")
print(f"Model used: {m2_output.get('model', 'unknown')}")
print(f"Generated at: {m2_output.get('generated_at', 'unknown')}")

print(f"\nSample documents:")
for i, cap in enumerate(captions[:3]):
    print(f"  [{i}] {cap.get('filename', '?')}: {cap.get('caption', '')[:80]}...")

In [ ]:
"""Generate embeddings from captions using sentence-transformers"""

MODEL_NAME = "all-MiniLM-L6-v2"  # 384-dim, fast on CPU
print(f"Loading embedding model: {MODEL_NAME}")

start_embed = time.time()
try:
    embed_model = SentenceTransformer(MODEL_NAME)
except Exception as e:
    raise RuntimeError(
        f"Failed to load the embedding model '{MODEL_NAME}': {e}\n"
        f"This model is public (no HF token needed) but must be downloaded once — "
        f"the instance needs outbound network egress to huggingface.co on the first "
        f"run. Check the workspace has internet egress, then re-run."
    ) from e
print(f"Model loaded in {time.time() - start_embed:.1f}s")

# Generate embeddings for all captions
texts = [cap["caption"] for cap in captions]
print(f"\nGenerating embeddings for {len(texts)} captions...")

start_encode = time.time()
embeddings = embed_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
encode_time = time.time() - start_encode

print(f"\nEmbedding generation complete:")
print(f"  Shape: {embeddings.shape}")
print(f"  Dimension: {embeddings.shape[1]}")
print(f"  Time: {encode_time:.2f}s")
print(f"  Per caption: {encode_time/len(texts)*1000:.1f}ms")

# Verify normalization
norms = np.linalg.norm(embeddings, axis=1)
print(f"  L2 norm range: [{norms.min():.4f}, {norms.max():.4f}] (should be ~1.0)")

In [ ]:
"""Create OpenSearch Serverless collection (or connect to pre-provisioned one)"""

def _iam_principal_arn():
    """AOSS data-access policies match on the IAM role/user ARN, NOT the STS
    assumed-role SESSION ARN. get_caller_identity returns the session form
    (arn:aws:sts::<acct>:assumed-role/<RoleName>/<session>); convert it to the
    IAM role ARN (arn:aws:iam::<acct>:role/<RoleName>) so index writes/searches
    under the SageMaker execution role are authorized instead of 403."""
    ident = boto3.client("sts").get_caller_identity()
    arn = ident["Arn"]
    acct = ident["Account"]
    if ":assumed-role/" in arn:
        role_name = arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
        return f"arn:aws:iam::{acct}:role/{role_name}"
    return arn  # already an iam user/role ARN


def get_or_create_collection(collection_name: str) -> dict:
    """Get existing collection or create a new VECTORSEARCH collection."""

    # Check if collection already exists. Narrow the except so a real
    # permission error surfaces instead of silently falling through to create.
    try:
        response = aoss_client.batch_get_collection(names=[collection_name])
        if response.get("collectionDetails"):
            collection = response["collectionDetails"][0]
            print(f"Found existing collection: {collection['name']}")
            print(f"  Status: {collection['status']}")
            print(f"  Endpoint: {collection.get('collectionEndpoint', 'N/A')}")
            return collection
    except aoss_client.exceptions.ResourceNotFoundException:
        pass
    except Exception as e:
        print(f"Collection lookup (continuing to create): {e}")

    # Create encryption policy (required before collection)
    encryption_policy = json.dumps({
        "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{collection_name}"]}],
        "AWSOwnedKey": True
    })

    try:
        aoss_client.create_security_policy(
            name=f"{collection_name}-enc",
            type="encryption",
            policy=encryption_policy
        )
        print("Created encryption policy.")
    except aoss_client.exceptions.ConflictException:
        print("Encryption policy already exists.")

    # Create network policy (public access for workshop)
    # Collection-only network rule (no public "dashboard"/web-UI rule).
    # Data access is still gated by IAM (aoss:APIAccessAll) + the data-
    # access policy naming this execution role, so the endpoint being
    # public does not permit unauthenticated reads/writes.
    network_policy = json.dumps([{
        "Rules": [{
            "ResourceType": "collection",
            "Resource": [f"collection/{collection_name}"]
        }],
        "AllowFromPublic": True
    }])

    try:
        aoss_client.create_security_policy(
            name=f"{collection_name}-net",
            type="network",
            policy=network_policy
        )
        print("Created network policy.")
    except aoss_client.exceptions.ConflictException:
        print("Network policy already exists.")

    # Create data access policy. Principal MUST be the IAM role ARN (not the STS
    # session ARN) or index writes/searches 403 under the execution role.
    caller_arn = _iam_principal_arn()
    print(f"Data-access principal: {caller_arn}")
    data_policy = json.dumps([{
        "Rules": [{
            "ResourceType": "index",
            "Resource": [f"index/{collection_name}/*"],
            "Permission": ["aoss:CreateIndex", "aoss:UpdateIndex", "aoss:DescribeIndex",
                           "aoss:ReadDocument", "aoss:WriteDocument"]
        }, {
            "ResourceType": "collection",
            "Resource": [f"collection/{collection_name}"],
            "Permission": ["aoss:CreateCollectionItems", "aoss:DescribeCollectionItems",
                           "aoss:UpdateCollectionItems"]
        }],
        "Principal": [caller_arn]
    }])

    try:
        aoss_client.create_access_policy(
            name=f"{collection_name}-access",
            type="data",
            policy=data_policy
        )
        print("Created data access policy.")
    except aoss_client.exceptions.ConflictException:
        print("Data access policy already exists.")

    # Create the collection
    print(f"\nCreating collection: {collection_name} (type=VECTORSEARCH)")
    try:
        response = aoss_client.create_collection(
            name=collection_name,
            type="VECTORSEARCH",
            description="AV 3.0 Blueprint Lab - Semantic video clip search"
        )
    except Exception as e:
        raise RuntimeError(
            f"Failed to create AOSS collection '{collection_name}': {e}\n"
            f"Common causes: the account AOSS collection quota is reached, or the "
            f"execution role lacks aoss:CreateCollection. Check Service Quotas "
            f"(OpenSearch Serverless) and the role policy."
        ) from e

    collection = response["createCollectionDetail"]
    print(f"  Collection ID: {collection['id']}")
    print(f"  Status: {collection['status']}")

    # Wait for collection to become ACTIVE
    print("\nWaiting for collection to become ACTIVE (this may take 2-5 minutes)...")
    for attempt in range(30):
        time.sleep(10)
        status_response = aoss_client.batch_get_collection(ids=[collection["id"]])
        current = status_response["collectionDetails"][0]
        status = current["status"]
        print(f"  [{attempt*10}s] Status: {status}")
        if status == "ACTIVE":
            print(f"\nCollection is ACTIVE!")
            print(f"  Endpoint: {current['collectionEndpoint']}")
            return current
        elif status == "FAILED":
            raise RuntimeError(f"Collection creation failed: {current}")

    raise TimeoutError("Collection did not become ACTIVE within 5 minutes.")

collection = get_or_create_collection(COLLECTION_NAME)
ENDPOINT = collection["collectionEndpoint"]

# AOSS data-access-policy changes propagate a few seconds after the collection is
# ACTIVE; a write/search issued immediately can still 403. Give it a moment.
time.sleep(15)

In [ ]:
"""Connect to OpenSearch Serverless and create k-NN vector index"""

# Build authenticated OpenSearch client
region = boto3.session.Session().region_name or "us-west-2"
credentials = boto3.Session().get_credentials()
if credentials is None:
    raise RuntimeError(
        "No AWS credentials available in this kernel — cannot sign OpenSearch "
        "requests. In SageMaker Studio the execution role is normally present; "
        "check the space's role configuration."
    )
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    "aoss",
    session_token=credentials.token
)

# Strip https:// prefix for host
host = ENDPOINT.replace("https://", "")

os_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=60
)

# Create k-NN index
index_body = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 512
        }
    },
    "mappings": {
        "properties": {
            "caption_embedding": {
                "type": "knn_vector",
                "dimension": EMBEDDING_DIM,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "faiss",
                    "parameters": {
                        "ef_construction": 512,
                        "m": 16
                    }
                }
            },
            "caption": {"type": "text"},
            "filename": {"type": "keyword"},
            "frame_idx": {"type": "integer"},
            "timestamp": {"type": "date"}
        }
    }
}

# Create the index. aoss does not support indices.exists (404), so we just try
# to create it and treat "already exists" as fine (aoss is fresh per collection,
# so this normally creates it outright).
#
# RETRY: a freshly-created collection's data-access policy takes time to
# propagate to the data plane — an index create issued too soon returns
# 403 "User does not have permissions for the requested resource" even though
# the policy names this exact role. That propagation can take well over the
# collection-cell's initial wait, so we retry the create with backoff and treat
# 403 as "policy still propagating" rather than a hard failure.
_created = False
_last_err = None
for _attempt in range(12):   # ~ up to ~2.5 min total (10s..~25s backoff, capped)
    try:
        os_client.indices.create(index=INDEX_NAME, body=index_body)
        _created = True
        break
    except Exception as _e:
        _msg = str(_e)
        if "resource_already_exists" in _msg or "already exists" in _msg:
            print(f"Index already exists: {INDEX_NAME}")
            _created = True
            break
        # 403 here == the data-access policy has not propagated yet; wait + retry.
        if "403" in _msg or "authorization_exception" in _msg or "does not have permissions" in _msg:
            _last_err = _e
            _wait = min(10 + _attempt * 3, 25)
            print(f"  [{_attempt+1}/12] AOSS policy still propagating (403) — "
                  f"retrying in {_wait}s...")
            time.sleep(_wait)
            continue
        raise   # any other error is a real failure — surface it
if not _created:
    raise RuntimeError(
        f"Could not create index '{INDEX_NAME}' after retries — the AOSS "
        f"data-access policy for this collection never propagated (last error "
        f"below). The policy principal is correct (see the collection cell); this "
        f"is a propagation delay. Wait a minute and re-run THIS cell.\n"
        f"  Last error: {_last_err}"
    )
print(f"Created k-NN index: {INDEX_NAME}")
print(f"  Dimension: {EMBEDDING_DIM}")
print(f"  Space type: cosinesimil")
print(f"  Engine: faiss (HNSW)")

In [ ]:
"""Index all caption embeddings into OpenSearch"""

print(f"Indexing {len(captions)} documents...")
start_index = time.time()

indexed_count = 0
errors = []

for i, (cap, embedding) in enumerate(zip(captions, embeddings)):
    try:
        doc = {
            "caption_embedding": embedding.tolist(),
            "caption": cap["caption"],
            "filename": cap["filename"],
            "frame_idx": cap.get("frame_idx", i),
            "timestamp": cap.get("timestamp", datetime.now(timezone.utc).isoformat()),
        }
        # AOSS (OpenSearch Serverless) does NOT support a caller-supplied
        # document _id on index/create — it auto-generates one. Passing id=
        # raises 400 illegal_argument_exception ("Document ID is not supported
        # in create/index operation request"). Omit it; AOSS assigns the _id.
        os_client.index(
            index=INDEX_NAME,
            body=doc,
        )
        indexed_count += 1
    except Exception as e:
        errors.append({"idx": i, "error": str(e)})

# NOTE: OpenSearch Serverless does NOT support the _refresh API
# (indices.refresh -> 404). aoss indexes documents automatically in
# near-real-time; wait briefly so the search cell sees them.
time.sleep(10)

index_time = time.time() - start_index
print(f"\nIndexing complete:")
print(f"  Documents indexed: {indexed_count}/{len(captions)}")
print(f"  Errors: {len(errors)}")
print(f"  Time: {index_time:.2f}s")
if indexed_count:
    print(f"  Throughput: {indexed_count/index_time:.1f} docs/s")

# Fail loudly if NOTHING indexed — otherwise the search cell silently returns
# zero hits with no clue. The most common cause is a data-access-policy 403
# (see the IAM-principal note in the collection cell).
if indexed_count == 0:
    sample = errors[0]["error"] if errors else "(no error captured)"
    raise RuntimeError(
        f"Indexed 0/{len(captions)} documents — the search demo would return "
        f"nothing. First error:\n  {sample}\n"
        f"If this is a 403/auth_error, the AOSS data-access policy principal may "
        f"not match this execution role — see the collection cell's IAM-principal "
        f"handling. If it is a connection/timeout, re-run after the collection has "
        f"settled."
    )

In [ ]:
"""Demo: Natural language queries → ranked video clip results"""

def semantic_search(query: str, k: int = 5) -> list:
    """Search for video clips using natural language."""
    # Encode query
    query_embedding = embed_model.encode([query], normalize_embeddings=True)[0]

    # k-NN search
    search_body = {
        "size": k,
        "query": {
            "knn": {
                "caption_embedding": {
                    "vector": query_embedding.tolist(),
                    "k": k
                }
            }
        },
        "_source": ["caption", "filename", "frame_idx"]
    }

    try:
        response = os_client.search(index=INDEX_NAME, body=search_body)
    except Exception as e:
        raise RuntimeError(
            f"OpenSearch query failed: {e}\n"
            f"If this is a 403/auth_error, the data-access policy principal may not "
            f"match this role; if it is 'index_not_found', indexing did not complete."
        ) from e

    results = []
    for hit in response["hits"]["hits"]:
        results.append({
            "score": hit["_score"],
            "filename": hit["_source"]["filename"],
            "frame_idx": hit["_source"].get("frame_idx"),
            "caption": hit["_source"]["caption"]
        })

    return results


# Demo queries representing AV use cases
DEMO_QUERIES = [
    "pedestrian crossing the road at an intersection",
    "highway driving with multiple lanes and vehicles ahead",
    "rainy weather conditions with reduced visibility",
    "parked cars on both sides of a narrow street",
    "traffic signal turning red with approaching vehicles"
]

print("=" * 70)
print("Semantic Video Clip Search — Demo Results")
print("=" * 70)

all_results = {}
for query in DEMO_QUERIES:
    print(f"\nQuery: \"{query}\"")
    print("-" * 50)

    results = semantic_search(query, k=3)
    all_results[query] = results

    for rank, r in enumerate(results, 1):
        print(f"  #{rank} [score={r['score']:.4f}] {r['filename']}")
        print(f"      {r['caption'][:100]}...")

print("\n" + "=" * 70)
print("Search demo complete. All queries executed via k-NN vector similarity.")

In [ ]:
"""Write index metadata and search results to S3"""

# Build output metadata
output_data = {
    "module": "M8_OpenSearch_Semantic_Search",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "collection_name": COLLECTION_NAME,
    "collection_endpoint": ENDPOINT,
    "index_name": INDEX_NAME,
    "embedding_model": MODEL_NAME,
    "embedding_dimension": EMBEDDING_DIM,
    "documents_indexed": indexed_count,
    "indexing_time_s": round(index_time, 2),
    "encoding_time_s": round(encode_time, 2),
    "demo_queries": [
        {
            "query": q,
            "results": [
                {"rank": i+1, "score": r["score"], "filename": r["filename"]}
                for i, r in enumerate(all_results[q])
            ]
        }
        for q in DEMO_QUERIES
    ]
}

# Upload metadata
output_key = f"{OUTPUT_PREFIX}index_metadata.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=output_key,
    Body=json.dumps(output_data, indent=2),
    ContentType="application/json"
)
print(f"Index metadata written to: s3://{USER_BUCKET}/{output_key}")

# Upload embeddings as numpy binary (for downstream use)
embeddings_key = f"{OUTPUT_PREFIX}embeddings.npy"
import io
buf = io.BytesIO()
np.save(buf, embeddings)
buf.seek(0)
s3.put_object(Bucket=USER_BUCKET, Key=embeddings_key, Body=buf.getvalue())
print(f"Embeddings written to: s3://{USER_BUCKET}/{embeddings_key}")

# Verify outputs
print(f"\nOutput Validation:")
for key in [output_key, embeddings_key]:
    head = s3.head_object(Bucket=USER_BUCKET, Key=key)
    print(f"  OK: {key} ({head['ContentLength']} bytes)")

In [ ]:
"""Cost Analysis — ml.t3.medium + OpenSearch Serverless"""

INSTANCE_TYPE = "ml.t3.medium"
INSTANCE_COST_PER_HOUR = 0.05  # USD, us-west-2 on-demand (adjust for your region)
KRW_RATE = 1370

# OpenSearch Serverless pricing
AOSS_OCU_COST_PER_HOUR = 0.24  # USD per OCU-hour
# AOSS bills a continuous per-account OCU floor 24/7. With standby
# replicas (the create default) the practical minimum is ~1 OCU for
# indexing + ~1 OCU for search = ~2 OCU while the collection exists.
AOSS_MIN_INDEXING_OCU = 1.0
AOSS_MIN_SEARCH_OCU = 1.0
AOSS_MONTHLY_MIN = (AOSS_MIN_INDEXING_OCU + AOSS_MIN_SEARCH_OCU) * AOSS_OCU_COST_PER_HOUR * 730  # hours/month

# Execution time estimate
estimated_total_min = 15  # encoding + indexing + queries
estimated_hours = estimated_total_min / 60

compute_cost_usd = INSTANCE_COST_PER_HOUR * estimated_hours
aoss_cost_per_hour = (AOSS_MIN_INDEXING_OCU + AOSS_MIN_SEARCH_OCU) * AOSS_OCU_COST_PER_HOUR
aoss_session_cost = aoss_cost_per_hour * estimated_hours  # only while active

total_session_cost_usd = compute_cost_usd + aoss_session_cost
total_session_cost_krw = total_session_cost_usd * KRW_RATE

print("=" * 60)
print("M8 OpenSearch Semantic Search — Cost Analysis")
print("=" * 60)
print(f"Instance type:       {INSTANCE_TYPE}")
print(f"Instance cost:       ${INSTANCE_COST_PER_HOUR:.2f}/hr")
print(f"Estimated time:      {estimated_total_min} min")
print(f"")
print(f"--- Compute ---")
print(f"  SageMaker:         ${compute_cost_usd:.3f} USD")
print(f"  OpenSearch (session): ${aoss_session_cost:.3f} USD")
print(f"  Session total:     ${total_session_cost_usd:.3f} USD ({total_session_cost_krw:.0f} KRW)")
print(f"")
print(f"--- OpenSearch Serverless Ongoing ---")
print(f"  Minimum OCUs:      {AOSS_MIN_INDEXING_OCU} indexing + {AOSS_MIN_SEARCH_OCU} search")
print(f"  Hourly minimum:    ${aoss_cost_per_hour:.2f}/hr")
print(f"  Monthly minimum:   ~${AOSS_MONTHLY_MIN:.0f}/month (while collection exists)")
print(f"  Monthly (KRW):     ~{AOSS_MONTHLY_MIN * KRW_RATE:,.0f} KRW/month")
print(f"")
print(f"--- vs NVIDIA Cosmos-embed NIM ---")
print(f"  NVAIE license:     order-of-thousands USD/yr per GPU node (list price varies)")
print(f"  OpenSearch (annual): ~${AOSS_MONTHLY_MIN * 12:,.0f}/yr")
print(f"  Savings:           Significant for workshop/dev use cases")
print(f"")
print(f"IMPORTANT: the collection bills ~24/7 until deleted. After the")
print(f"workshop, delete the collection AND its 3 policies to stop charges:")
print(f"  aws opensearchserverless delete-collection --id {collection['id']}")
print(f"  aws opensearchserverless delete-security-policy --name {COLLECTION_NAME}-enc --type encryption")
print(f"  aws opensearchserverless delete-security-policy --name {COLLECTION_NAME}-net --type network")
print(f"  aws opensearchserverless delete-access-policy   --name {COLLECTION_NAME}-access --type data")
print(f"  (or rely on user teardown / the workshop teardown script)")
print("=" * 60)

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m08-opensearch")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")